In [13]:
# Install all required packages
!pip install kafka-python minio pandas numpy scipy faker --quiet

In [2]:
# Imports & Core Utilities
import json
import csv
import math
import statistics
import io
import time
import threading
import warnings
from datetime import datetime
from collections import Counter
from typing import Any
from pathlib import Path

import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
from faker import Faker

warnings.filterwarnings("ignore")
fake = Faker()
print("✅ Imports complete")

✅ Imports complete


In [3]:
# Type Inference Engine

def _try_numeric(value: str):
    try:
        return float(str(value).replace(",", "").strip())
    except (ValueError, AttributeError):
        return None

def _try_datetime(value: str):
    formats = [
        "%Y-%m-%d", "%d/%m/%Y", "%m/%d/%Y",
        "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S",
        "%d-%m-%Y", "%B %d, %Y",
    ]
    for fmt in formats:
        try:
            return datetime.strptime(str(value).strip(), fmt)
        except (ValueError, AttributeError):
            continue
    return None

def infer_column_type(values: list) -> str:
    """
    Infer semantic type from raw string values.
    Priority: boolean → integer → float → datetime → string
    """
    non_null = [v for v in values if str(v).strip() not in ("", "None", "nan", "NaN")]
    if not non_null:
        return "empty"

    bool_tokens = {"true", "false", "yes", "no", "1", "0"}
    if all(str(v).strip().lower() in bool_tokens for v in non_null):
        return "boolean"

    numerics = [_try_numeric(v) for v in non_null]
    if all(n is not None for n in numerics):
        if all(float(n).is_integer() for n in numerics):
            return "integer"
        return "float"

    if all(_try_datetime(v) is not None for v in non_null):
        return "datetime"

    return "string"

print("✅ Type inference engine ready")

✅ Type inference engine ready


In [4]:
# Statistical Analysis Functions

def _percentile(sorted_data: list, p: float) -> float:
    if not sorted_data:
        return float("nan")
    n = len(sorted_data)
    idx = (p / 100) * (n - 1)
    lo, hi = int(idx), min(int(idx) + 1, n - 1)
    return sorted_data[lo] + (idx - lo) * (sorted_data[hi] - sorted_data[lo])


def numeric_stats(values: list) -> dict:
    """Full descriptive statistics for a numeric column."""
    if not values:
        return {}

    n = len(values)
    sorted_vals = sorted(values)
    mean = statistics.mean(values)
    variance = statistics.variance(values) if n > 1 else 0.0
    std_dev = math.sqrt(variance)

    # Skewness (Fisher's moment)
    skewness = 0.0
    if std_dev > 0 and n > 2:
        skewness = (n / ((n - 1) * (n - 2))) * sum(
            ((v - mean) / std_dev) ** 3 for v in values
        )

    # Kurtosis (excess)
    kurtosis = 0.0
    if std_dev > 0 and n > 3:
        kurtosis = (
            n * (n + 1) / ((n - 1) * (n - 2) * (n - 3))
            * sum(((v - mean) / std_dev) ** 4 for v in values)
            - 3 * (n - 1) ** 2 / ((n - 2) * (n - 3))
        )

    # IQR-based outlier detection
    q1 = _percentile(sorted_vals, 25)
    q3 = _percentile(sorted_vals, 75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outliers = [v for v in values if v < lower_fence or v > upper_fence]

    # Normality test (Shapiro-Wilk, max 5000 samples)
    sample = values[:5000]
    try:
        shapiro_stat, shapiro_p = scipy_stats.shapiro(sample)
    except Exception:
        shapiro_stat, shapiro_p = float("nan"), float("nan")

    # Distribution histogram (10 bins)
    hist, bin_edges = np.histogram(values, bins=10)
    histogram = [
        {"bin_start": round(float(bin_edges[i]), 4),
         "bin_end":   round(float(bin_edges[i + 1]), 4),
         "count":     int(hist[i])}
        for i in range(len(hist))
    ]

    return {
        "count":          n,
        "mean":           round(mean, 6),
        "median":         round(_percentile(sorted_vals, 50), 6),
        "std_dev":        round(std_dev, 6),
        "variance":       round(variance, 6),
        "min":            sorted_vals[0],
        "max":            sorted_vals[-1],
        "range":          round(sorted_vals[-1] - sorted_vals[0], 6),
        "p25":            round(q1, 6),
        "p50":            round(_percentile(sorted_vals, 50), 6),
        "p75":            round(q3, 6),
        "p95":            round(_percentile(sorted_vals, 95), 6),
        "iqr":            round(iqr, 6),
        "skewness":       round(skewness, 6),
        "kurtosis":       round(kurtosis, 6),
        "outlier_count":  len(outliers),
        "outlier_pct":    round(len(outliers) / n * 100, 2),
        "outlier_values": sorted(outliers)[:20],
        "normality_shapiro_stat": round(float(shapiro_stat), 6),
        "normality_shapiro_p":    round(float(shapiro_p), 6),
        "is_normal_dist":         bool(shapiro_p > 0.05),
        "histogram":      histogram,
    }


def categorical_stats(values: list) -> dict:
    """Frequency analysis for categorical / string columns."""
    n = len(values)
    counter = Counter(values)
    top_10 = counter.most_common(10)
    return {
        "count":        n,
        "unique_count": len(counter),
        "top_values":   [{"value": v, "count": c, "pct": round(c / n * 100, 2)} for v, c in top_10],
        "least_common": [{"value": v, "count": c} for v, c in counter.most_common()[:-6:-1]],
        "entropy":      round(scipy_stats.entropy(list(counter.values()), base=2), 6) if counter else 0,
    }


def datetime_stats(values: list) -> dict:
    """Date range and span for datetime columns."""
    timestamps = sorted(v.timestamp() for v in values)
    min_dt = datetime.fromtimestamp(min(timestamps))
    max_dt = datetime.fromtimestamp(max(timestamps))
    return {
        "count":     len(values),
        "min":       min_dt.isoformat(),
        "max":       max_dt.isoformat(),
        "span_days": (max_dt - min_dt).days,
    }


def profile_column(col_name: str, raw_values: list, inferred_type: str) -> dict:
    """Dispatch statistics to the correct analyzer."""
    non_null = [v for v in raw_values if str(v).strip() not in ("", "None", "nan", "NaN")]

    if inferred_type in ("integer", "float"):
        nums = [_try_numeric(v) for v in non_null if _try_numeric(v) is not None]
        stats = numeric_stats(nums)
    elif inferred_type == "datetime":
        dts = [_try_datetime(v) for v in non_null if _try_datetime(v) is not None]
        stats = datetime_stats(dts)
    elif inferred_type == "boolean":
        stats = categorical_stats([str(v).strip().lower() for v in non_null])
    else:
        stats = categorical_stats([str(v) for v in non_null])

    return {"column": col_name, "type": inferred_type, "statistics": stats}


print("✅ Statistical analysis functions ready")

✅ Statistical analysis functions ready


In [5]:
#  Metadata Extraction & Schema Inference

def extract_metadata(data: list[dict]) -> dict:
    """
    Extract full schema metadata from a list of row-dicts.
    Returns row_count, column_count, and per-column metadata.
    """
    if not data:
        return {"row_count": 0, "column_count": 0, "columns": {}}

    # Collect raw column values
    columns: dict[str, list] = {}
    for row in data:
        for col, val in row.items():
            columns.setdefault(col, []).append(str(val) if val is not None else "")

    schema = {}
    for col, raw_values in columns.items():
        inferred_type = infer_column_type(raw_values)
        null_count = sum(1 for v in raw_values if str(v).strip() in ("", "None", "nan", "NaN"))
        unique_values = set(raw_values)

        # Sample values (first 5 non-null)
        samples = [v for v in raw_values if str(v).strip() not in ("", "None")][:5]

        schema[col] = {
            "inferred_type":  inferred_type,
            "total_count":    len(raw_values),
            "null_count":     null_count,
            "non_null_count": len(raw_values) - null_count,
            "null_pct":       round(null_count / len(raw_values) * 100, 2) if raw_values else 0,
            "unique_count":   len(unique_values),
            "is_unique":      len(unique_values) == len(raw_values),
            "sample_values":  samples,
        }

    return {
        "row_count":    len(data),
        "column_count": len(columns),
        "columns":      schema,
    }


def compute_correlations(data: list[dict], numeric_cols: list[str]) -> dict:
    """Pearson correlation matrix for all numeric column pairs."""
    if len(numeric_cols) < 2:
        return {}

    vectors = {}
    for col in numeric_cols:
        vals = []
        for row in data:
            n = _try_numeric(str(row.get(col, "")))
            vals.append(n if n is not None else float("nan"))
        vectors[col] = vals

    clean_idx = [
        i for i in range(len(data))
        if all(not math.isnan(vectors[c][i]) for c in numeric_cols)
    ]
    clean = {c: [vectors[c][i] for i in clean_idx] for c in numeric_cols}

    matrix = {}
    for a in numeric_cols:
        matrix[a] = {}
        for b in numeric_cols:
            if a == b:
                matrix[a][b] = 1.0
                continue
            xa, xb = clean[a], clean[b]
            n = len(xa)
            if n < 2:
                matrix[a][b] = float("nan")
                continue
            mean_a = statistics.mean(xa)
            mean_b = statistics.mean(xb)
            num  = sum((xa[i] - mean_a) * (xb[i] - mean_b) for i in range(n))
            den_a = math.sqrt(sum((v - mean_a) ** 2 for v in xa))
            den_b = math.sqrt(sum((v - mean_b) ** 2 for v in xb))
            denom = den_a * den_b
            matrix[a][b] = round(num / denom, 6) if denom else float("nan")

    return matrix


print("✅ Metadata extraction & schema inference ready")

✅ Metadata extraction & schema inference ready


In [6]:
# Full Profile Report Builder
def profile_dataset(data: list[dict], dataset_name: str = "dataset", source: str = "memory") -> dict:
    """
    Run the complete profiling pipeline.

    Parameters
    ----------
    data         : list of row-dicts
    dataset_name : label for the report
    source       : origin label e.g. 'csv', 'kafka', 'minio'
    """
    start = time.time()
    metadata     = extract_metadata(data)
    columns_meta = metadata["columns"]

    column_profiles = []
    numeric_cols    = []

    for col, meta in columns_meta.items():
        raw_values = [str(row.get(col, "")) for row in data]
        profile    = profile_column(col, raw_values, meta["inferred_type"])
        profile["metadata"] = meta
        column_profiles.append(profile)
        if meta["inferred_type"] in ("integer", "float"):
            numeric_cols.append(col)

    correlations = compute_correlations(data, numeric_cols)
    elapsed      = round(time.time() - start, 4)

    return {
        "report_meta": {
            "dataset_name":  dataset_name,
            "data_source":   source,
            "generated_at":  datetime.utcnow().isoformat() + "Z",
            "row_count":     metadata["row_count"],
            "column_count":  metadata["column_count"],
            "profiling_time_sec": elapsed,
        },
        "schema":          columns_meta,
        "column_profiles": column_profiles,
        "correlations":    correlations,
    }

print("✅ Profile report builder ready")

✅ Profile report builder ready


In [7]:
# Catalog Output — JSON & CSV

def save_catalog_json(report: dict, path: str = "catalog.json") -> None:
    """Write the full catalog report to a JSON file."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, default=str)
    print(f"✅ JSON catalog saved → {path}")


def save_catalog_csv(report: dict, path: str = "catalog_summary.csv") -> None:
    """Write a flat one-row-per-column CSV catalog summary."""
    fieldnames = [
        "dataset", "source", "column", "inferred_type",
        "total_count", "null_count", "null_pct", "unique_count", "is_unique",
        "mean", "median", "std_dev", "min", "max", "range",
        "p25", "p75", "p95", "skewness", "kurtosis",
        "outlier_count", "outlier_pct", "is_normal_dist",
        "top_value", "top_value_count", "entropy",
        "sample_values",
    ]

    rows = []
    meta = report["report_meta"]
    for cp in report["column_profiles"]:
        m  = cp.get("metadata", {})
        st = cp.get("statistics", {})
        top = st.get("top_values", [])
        row = {
            "dataset":        meta["dataset_name"],
            "source":         meta.get("data_source", ""),
            "column":         cp["column"],
            "inferred_type":  cp["type"],
            "total_count":    m.get("total_count", ""),
            "null_count":     m.get("null_count", ""),
            "null_pct":       m.get("null_pct", ""),
            "unique_count":   m.get("unique_count", ""),
            "is_unique":      m.get("is_unique", ""),
            "sample_values":  "|".join(str(v) for v in m.get("sample_values", [])),
        }
        for field in ("mean", "median", "std_dev", "min", "max", "range",
                      "p25", "p75", "p95", "skewness", "kurtosis",
                      "outlier_count", "outlier_pct", "is_normal_dist", "entropy"):
            row[field] = st.get(field, "")
        row["top_value"]       = top[0]["value"] if top else ""
        row["top_value_count"] = top[0]["count"] if top else ""
        rows.append(row)

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"✅ CSV catalog saved  → {path}")


def display_catalog_summary(report: dict) -> None:
    """Pretty-print a quick summary table in Jupyter."""
    meta = report["report_meta"]
    print(f"\n{'═'*65}")
    print(f"  CATALOG REPORT — {meta['dataset_name']}  [{meta['data_source']}]")
    print(f"  Generated : {meta['generated_at']}")
    print(f"  Rows: {meta['row_count']:,}   Columns: {meta['column_count']}   "
          f"Profiled in {meta['profiling_time_sec']}s")
    print(f"{'═'*65}")
    print(f"  {'Column':<20} {'Type':<12} {'Nulls%':>6}  {'Unique':>7}  {'Key Stat'}")
    print(f"  {'─'*60}")
    for cp in report["column_profiles"]:
        m  = cp["metadata"]
        st = cp["statistics"]
        if cp["type"] in ("integer", "float"):
            key_stat = f"mean={st.get('mean', 'N/A')}"
        elif cp["type"] == "datetime":
            key_stat = f"span={st.get('span_days', 'N/A')}d"
        else:
            top = st.get("top_values", [])
            key_stat = f"top='{top[0]['value']}'" if top else ""
        print(f"  {cp['column']:<20} {cp['type']:<12} {m['null_pct']:>5}%  "
              f"{m['unique_count']:>7}  {key_stat}")
    print(f"{'═'*65}\n")

print("✅ Catalog output functions ready")

✅ Catalog output functions ready


In [8]:
# Kafka Streaming Profiler

# ── Mock Kafka producer (no broker needed for testing) ──────────────────────

class MockKafkaConsumer:
    """Simulates a Kafka consumer by generating random JSON messages."""
    def __init__(self, topic: str, n_messages: int = 200):
        self.topic     = topic
        self.n_messages = n_messages
        self._pos      = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self._pos >= self.n_messages:
            raise StopIteration
        self._pos += 1

        msg = {
            "user_id":    fake.random_int(1, 1000),
            "event":      fake.random_element(["click","view","purchase","signup"]),
            "amount":     round(fake.pyfloat(min_value=0, max_value=5000), 2),
            "timestamp":  fake.date_time_this_year().strftime("%Y-%m-%dT%H:%M:%S"),
            "region":     fake.random_element(["NA","EU","APAC","LATAM"]),
            "active":     str(fake.boolean()).lower(),
        }
        return type("Msg", (), {"value": json.dumps(msg).encode()})()

    def close(self):
        pass


def profile_kafka_stream(
    topic: str,
    bootstrap_servers: str = None,   # None → use mock consumer
    batch_size: int = 100,
    max_messages: int = 500,
    group_id: str = "profiler-group",
) -> dict:
    """
    Profile a Kafka topic by consuming up to max_messages in batches.

    Parameters
    ----------
    topic             : Kafka topic name
    bootstrap_servers : 'host:port' — leave None to use the built-in mock
    batch_size        : rows accumulated before each incremental profile update
    max_messages      : stop after this many messages
    group_id          : Kafka consumer group id

    Returns
    -------
    Full profile report dict (same schema as profile_dataset)
    """
    print(f"🔄 Connecting to Kafka topic '{topic}' "
          f"({'MOCK' if not bootstrap_servers else bootstrap_servers}) …")

    if bootstrap_servers:
        from kafka import KafkaConsumer  # real consumer
        consumer = KafkaConsumer(
            topic,
            bootstrap_servers=bootstrap_servers,
            auto_offset_reset="earliest",
            group_id=group_id,
            value_deserializer=lambda m: json.loads(m.decode("utf-8")),
            consumer_timeout_ms=5000,
        )
    else:
        consumer = MockKafkaConsumer(topic, n_messages=max_messages)

    buffer: list[dict] = []
    count = 0

    for msg in consumer:
        try:
            if bootstrap_servers:
                row = msg.value
            else:
                row = json.loads(msg.value.decode())
            buffer.append(row)
            count += 1
        except Exception as e:
            print(f"  ⚠ Skipped malformed message: {e}")
            continue

        if count >= max_messages:
            break

    consumer.close()
    print(f"  📨 Consumed {count} messages from '{topic}'")

    if not buffer:
        print("  ⚠ No messages received.")
        return {}

    report = profile_dataset(buffer, dataset_name=f"kafka::{topic}", source="kafka")
    return report


print("✅ Kafka streaming profiler ready")

✅ Kafka streaming profiler ready


In [9]:
# MinIO Object-Store Profiler
class MockMinIOClient:
    """
    Simulates MinIO by holding in-memory CSV objects.
    Generates two synthetic datasets automatically.
    """
    def __init__(self):
        self._objects: dict[str, bytes] = {}
        self._generate_mock_data()

    def _generate_mock_data(self):
        # Dataset 1: transactions
        rows1 = [{"txn_id": i,
                  "customer": fake.name(),
                  "amount": round(fake.pyfloat(min_value=5, max_value=10000), 2),
                  "currency": fake.currency_code(),
                  "date": fake.date_this_decade().isoformat(),
                  "status": fake.random_element(["completed","pending","failed"])}
                 for i in range(1, 301)]
        buf1 = io.StringIO()
        w = csv.DictWriter(buf1, fieldnames=rows1[0].keys())
        w.writeheader(); w.writerows(rows1)
        self._objects["transactions.csv"] = buf1.getvalue().encode()

        # Dataset 2: sensors
        rows2 = [{"sensor_id": f"S{i:04d}",
                  "temperature": round(fake.pyfloat(min_value=-10, max_value=120), 2),
                  "humidity": round(fake.pyfloat(min_value=0, max_value=100), 2),
                  "reading_ts": fake.date_time_this_year().strftime("%Y-%m-%dT%H:%M:%S"),
                  "unit": fake.random_element(["A","B","C"])}
                 for i in range(1, 201)]
        buf2 = io.StringIO()
        w = csv.DictWriter(buf2, fieldnames=rows2[0].keys())
        w.writeheader(); w.writerows(rows2)
        self._objects["sensor_readings.csv"] = buf2.getvalue().encode()

    def list_objects(self, bucket: str, prefix: str = ""):
        return [type("Obj", (), {"object_name": k})()
                for k in self._objects if k.startswith(prefix)]

    def get_object(self, bucket: str, name: str):
        data = self._objects.get(name, b"")
        return type("Resp", (), {"read": lambda self: data, "data": data})()


def _read_minio_object(client, bucket: str, name: str) -> list[dict]:
    """Download and parse a CSV object from MinIO (real or mock)."""
    try:
        response = client.get_object(bucket, name)
        raw = response.read() if callable(response.read) else response.data
        text = raw.decode("utf-8")
        reader = csv.DictReader(io.StringIO(text))
        return list(reader)
    except Exception as e:
        print(f"  ⚠ Could not read '{name}': {e}")
        return []


def profile_minio_bucket(
    bucket: str,
    endpoint: str = None,       # None → use mock client
    access_key: str = None,
    secret_key: str = None,
    prefix: str = "",
    secure: bool = False,
) -> list[dict]:
    """
    Profile all CSV objects in a MinIO bucket.

    Parameters
    ----------
    bucket     : bucket name
    endpoint   : 'host:port' — leave None to use the built-in mock
    access_key : MinIO access key
    secret_key : MinIO secret key
    prefix     : optional object name prefix filter
    secure     : use HTTPS if True

    Returns
    -------
    List of profile report dicts, one per object
    """
    if endpoint:
        from minio import Minio
        client = Minio(endpoint, access_key=access_key,
                       secret_key=secret_key, secure=secure)
        print(f"🗄  Connected to MinIO at {endpoint}, bucket='{bucket}'")
    else:
        client = MockMinIOClient()
        print(f"🗄  Using MOCK MinIO client, bucket='{bucket}'")

    objects = client.list_objects(bucket, prefix=prefix)
    reports = []

    for obj in objects:
        name = obj.object_name
        if not name.endswith(".csv"):
            print(f"  ⏭ Skipping non-CSV object: {name}")
            continue
        print(f"  📂 Profiling: {name}")
        data = _read_minio_object(client, bucket, name)
        if data:
            report = profile_dataset(data, dataset_name=name, source="minio")
            reports.append(report)

    print(f"✅ Profiled {len(reports)} objects from bucket '{bucket}'")
    return reports

print("✅ MinIO object-store profiler ready")

✅ MinIO object-store profiler ready


In [11]:
# Accuracy Test Suite
# ── Known ground-truth dataset ───────────────────────────────────────────────
KNOWN_DATA = [
    {"id":"1",  "name":"Alice",  "age":"30","salary":"70000.00","joined":"2020-01-15","active":"true"},
    {"id":"2",  "name":"Bob",    "age":"25","salary":"50000.50","joined":"2019-03-22","active":"false"},
    {"id":"3",  "name":"Carol",  "age":"35","salary":"90000.75","joined":"2021-07-01","active":"true"},
    {"id":"4",  "name":"Dave",   "age":"28","salary":"60000.00","joined":"2018-11-30","active":"true"},
    {"id":"5",  "name":"Eve",    "age":"40","salary":"120000.00","joined":"2017-05-10","active":"false"},
    {"id":"6",  "name":"Frank",  "age":"22","salary":"45000.00","joined":"2022-02-28","active":"true"},
    {"id":"7",  "name":"Grace",  "age":"33","salary":"80000.00","joined":"2020-09-14","active":"true"},
    {"id":"8",  "name":"Hank",   "age":"55","salary":"150000.00","joined":"2015-04-05","active":"false"},
    {"id":"9",  "name":"Ivy",    "age":"29","salary":"62000.00","joined":"2019-12-01","active":"true"},
    {"id":"10", "name":"Jack",   "age":"31","salary":"71000.00","joined":"2021-03-18","active":"false"},
    {"id":"11", "name":"",       "age":"27","salary":"",         "joined":"2023-01-01","active":"true"},
]

ages    = [30,25,35,28,40,22,33,55,29,31,27]
salaries= [70000,50000.50,90000.75,60000,120000,45000,80000,150000,62000,71000]

EXPECTED = {
    "age_mean":          round(statistics.mean(ages), 6),
    "age_min":           22,
    "age_max":           55,
    "age_range":         33,
    "age_count":         11,
    "name_null_count":   1,
    "salary_null_count": 1,
    "row_count":         11,
    "column_count":      6,
}


def run_accuracy_tests(verbose: bool = True) -> dict:
    """
    Run accuracy tests against the known dataset.
    Returns a summary dict with passed / failed / total counts.
    """
    report  = profile_dataset(KNOWN_DATA, "known_dataset")
    profiles= {cp["column"]: cp for cp in report["column_profiles"]}
    results = []

    def check(name: str, actual, expected, tol: float = 0.001):
        try:
            if isinstance(expected, float):
                ok = abs(actual - expected) <= tol
            elif isinstance(expected, bool):
                ok = actual == expected
            else:
                ok = actual == expected
        except Exception:
            ok = False
        results.append({"test": name, "passed": ok, "actual": actual, "expected": expected})
        if verbose:
            status = "✅ PASS" if ok else "❌ FAIL"
            print(f"  {status}  {name:<45}  actual={actual}  expected={expected}")

    print("\n" + "─"*70)
    print(" ACCURACY TEST SUITE")
    print("─"*70)

    # ── Type Inference ──────────────────────────────────────────────────────
    print("\n[Type Inference]")
    check("id inferred as integer",  profiles["id"]["type"],     "integer")
    check("name inferred as string", profiles["name"]["type"],   "string")
    check("age inferred as integer", profiles["age"]["type"],    "integer")
    check("salary inferred as float",profiles["salary"]["type"], "float")
    check("joined inferred as datetime", profiles["joined"]["type"], "datetime")
    check("active inferred as boolean",  profiles["active"]["type"], "boolean")

    # ── Null Detection ──────────────────────────────────────────────────────
    print("\n[Null Detection]")
    check("name null_count",   profiles["name"]["metadata"]["null_count"],   EXPECTED["name_null_count"])
    check("salary null_count", profiles["salary"]["metadata"]["null_count"], EXPECTED["salary_null_count"])
    check("age null_count",    profiles["age"]["metadata"]["null_count"],    0)

    # ── Numeric Stats ───────────────────────────────────────────────────────
    print("\n[Numeric Statistics]")
    age_st = profiles["age"]["statistics"]
    check("age mean",   age_st["mean"],   EXPECTED["age_mean"],  tol=0.01)
    check("age min",    age_st["min"],    EXPECTED["age_min"])
    check("age max",    age_st["max"],    EXPECTED["age_max"])
    check("age range",  age_st["range"],  EXPECTED["age_range"])
    check("age count",  age_st["count"],  EXPECTED["age_count"])
    check("age median", age_st["median"], _percentile(sorted(ages), 50), tol=0.01)
    check("age p75 >= p25", age_st["p75"] >= age_st["p25"], True)
    check("age std_dev > 0", age_st["std_dev"] > 0, True)

    # ── Outlier Detection ───────────────────────────────────────────────────
    print("\n[Outlier Detection]")
    check("Hank (55) detected as outlier", 55.0 in age_st["outlier_values"], True)

    # ── Uniqueness ──────────────────────────────────────────────────────────
    print("\n[Uniqueness]")
    check("id is_unique",      profiles["id"]["metadata"]["is_unique"],     True)
    check("active not unique", profiles["active"]["metadata"]["is_unique"], False)

    # ── Correlations ────────────────────────────────────────────────────────
    print("\n[Correlations]")
    corr = report["correlations"]
    check("age self-correlation = 1.0", corr.get("age", {}).get("age"), 1.0, tol=1e-5)
    if "salary" in corr.get("age", {}):
        check("correlation symmetric",
              corr["age"]["salary"], corr["salary"]["age"], tol=1e-5)

    # ── Schema ──────────────────────────────────────────────────────────────
    print("\n[Schema Summary]")
    check("row_count",    report["report_meta"]["row_count"],    EXPECTED["row_count"])
    check("column_count", report["report_meta"]["column_count"], EXPECTED["column_count"])

    # ── Results summary ─────────────────────────────────────────────────────
    passed = sum(1 for r in results if r["passed"])
    total  = len(results)
    failed = total - passed

    print("\n" + "─"*70)
    print(f" RESULTS:  {passed}/{total} passed   {failed} failed")
    print("─"*70 + "\n")

    return {"passed": passed, "failed": failed, "total": total, "details": results}


test_summary = run_accuracy_tests()


──────────────────────────────────────────────────────────────────────
 ACCURACY TEST SUITE
──────────────────────────────────────────────────────────────────────

[Type Inference]
  ✅ PASS  id inferred as integer                         actual=integer  expected=integer
  ✅ PASS  name inferred as string                        actual=string  expected=string
  ✅ PASS  age inferred as integer                        actual=integer  expected=integer
  ✅ PASS  salary inferred as float                       actual=float  expected=float
  ✅ PASS  joined inferred as datetime                    actual=datetime  expected=datetime
  ✅ PASS  active inferred as boolean                     actual=boolean  expected=boolean

[Null Detection]
  ✅ PASS  name null_count                                actual=1  expected=1
  ✅ PASS  salary null_count                              actual=1  expected=1
  ✅ PASS  age null_count                                 actual=0  expected=0

[Numeric Statistics]
  ✅ PASS

In [14]:
# Run Everything End-to-End
print("=" * 65)
print("  STEP 1 — Profile Kafka Stream (mock)")
print("=" * 65)
kafka_report = profile_kafka_stream(topic="user-events", max_messages=300)
if kafka_report:
    display_catalog_summary(kafka_report)
    save_catalog_json(kafka_report, "kafka_catalog.json")
    save_catalog_csv(kafka_report,  "kafka_catalog_summary.csv")


print("\n" + "=" * 65)
print("  STEP 2 — Profile MinIO Bucket (mock)")
print("=" * 65)
minio_reports = profile_minio_bucket(bucket="data-lake")
for r in minio_reports:
    display_catalog_summary(r)
save_catalog_json({"reports": minio_reports}, "minio_catalog.json")


print("\n" + "=" * 65)
print("  STEP 3 — Accuracy Tests")
print("=" * 65)
# (already ran in Cell 10 — re-running for completeness)
run_accuracy_tests(verbose=False)

print("\n✅ All done! Files generated:")
for f in ["kafka_catalog.json", "kafka_catalog_summary.csv", "minio_catalog.json"]:
    try:
        size = Path(f).stat().st_size
        print(f"  📄 {f}  ({size:,} bytes)")
    except FileNotFoundError:
        pass

  STEP 1 — Profile Kafka Stream (mock)
🔄 Connecting to Kafka topic 'user-events' (MOCK) …
  📨 Consumed 300 messages from 'user-events'

═════════════════════════════════════════════════════════════════
  CATALOG REPORT — kafka::user-events  [kafka]
  Generated : 2026-04-26T16:41:55.258292Z
  Rows: 300   Columns: 6   Profiled in 0.0385s
═════════════════════════════════════════════════════════════════
  Column               Type         Nulls%   Unique  Key Stat
  ────────────────────────────────────────────────────────────
  user_id              integer        0.0%      260  mean=472.513333
  event                string         0.0%        4  top='view'
  amount               float          0.0%      300  mean=2401.568867
  timestamp            datetime       0.0%      300  span=115d
  region               string         0.0%        4  top='LATAM'
  active               boolean        0.0%        2  top='true'
═════════════════════════════════════════════════════════════════

✅ JSON ca